In [1]:
import duckdb
import time

# Connect to a DuckDB database file
# This creates a persistent file we can reopen later instead of re-running everything
con = duckdb.connect("../data/spotify.duckdb")

# Count the rows in the CSV directly, no need to load it into memory
# DuckDB scans the file efficiently
start = time.time()

result = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto('../data/charts.csv')
""").fetchone()

elapsed = time.time() - start
print(f"Total rows: {result[0]:,}")
print(f"Time: {elapsed:.1f} seconds")

Total rows: 26,173,514
Time: 1.0 seconds


In [2]:
# Import the CSV into a real DuckDB table named "charts"
# After this, queries run against the table directly, much faster than re-reading the CSV
start = time.time()

con.execute("""
    CREATE OR REPLACE TABLE charts AS
    SELECT * FROM read_csv_auto('../data/charts.csv')
""")

elapsed = time.time() - start
print(f"Import time: {elapsed:.1f} seconds")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Import time: 6.5 seconds


In [3]:
# What columns do we have?
print(con.execute("DESCRIBE charts").df())

  column_name column_type null   key default extra
0       title     VARCHAR  YES  None    None  None
1        rank      BIGINT  YES  None    None  None
2        date        DATE  YES  None    None  None
3      artist     VARCHAR  YES  None    None  None
4         url     VARCHAR  YES  None    None  None
5      region     VARCHAR  YES  None    None  None
6       chart     VARCHAR  YES  None    None  None
7       trend     VARCHAR  YES  None    None  None
8     streams      BIGINT  YES  None    None  None


In [4]:
# Sample some rows to see what the data actually looks like
print(con.execute("SELECT * FROM charts LIMIT 5").df())

                         title  rank       date  \
0      Chantaje (feat. Maluma)     1 2017-01-01   
1  Vente Pa' Ca (feat. Maluma)     2 2017-01-01   
2   Reggaetón Lento (Bailemos)     3 2017-01-01   
3                       Safari     4 2017-01-01   
4                  Shaky Shaky     5 2017-01-01   

                                  artist  \
0                                Shakira   
1                           Ricky Martin   
2                                   CNCO   
3  J Balvin, Pharrell Williams, BIA, Sky   
4                           Daddy Yankee   

                                                 url     region   chart  \
0  https://open.spotify.com/track/6mICuAdrwEjh6Y6...  Argentina  top200   
1  https://open.spotify.com/track/7DM4BPaS7uofFul...  Argentina  top200   
2  https://open.spotify.com/track/3AEZUABDXNtecAO...  Argentina  top200   
3  https://open.spotify.com/track/6rQSrBHf7HlZjtc...  Argentina  top200   
4  https://open.spotify.com/track/58IL315gMSTD37D... 

In [5]:
# How many distinct countries (regions) are in the data?
# How many days does the data cover?
result = con.execute("""
    SELECT 
        COUNT(DISTINCT region) AS countries,
        MIN(date) AS earliest,
        MAX(date) AS latest,
        COUNT(DISTINCT date) AS days_covered
    FROM charts
""").df()
print(result)

   countries   earliest     latest  days_covered
0         70 2017-01-01 2021-12-31          1826


In [10]:
# Run Question 1: market concentration by country
result = con.execute(open("../sql/01_market_concentration.sql").read()).df()
print(result.head(15))
print("...")
print(result.tail(15))

           region  top10_appearances  total_appearances  pct_from_top10
0           Japan            89272.0           356774.0           25.02
1         Iceland            44812.0           212123.0           21.13
2       Singapore            74142.0           358388.0           20.69
3     New Zealand            71013.0           358387.0           19.81
4   United States            71733.0           364184.0           19.70
5     Philippines            65540.0           358390.0           18.29
6       Hong Kong            65371.0           358388.0           18.24
7         Estonia            19231.0           108007.0           17.81
8          Brazil            64571.0           364516.0           17.71
9        Malaysia            61984.0           358390.0           17.30
10         France            61933.0           358390.0           17.28
11         Canada            62316.0           361390.0           17.24
12       Honduras            48664.0           286269.0         

In [9]:
# Save the result so we can use it later for charts and the dashboard
result.to_csv("../data/01_market_concentration.csv", index=False)
print("Saved to data/01_market_concentration.csv")

Saved to data/01_market_concentration.csv


In [8]:
# Check which countries have too little data to compare fairly
result_with_days = con.execute("""
    SELECT 
        region,
        COUNT(DISTINCT date) AS days_with_data
    FROM charts
    WHERE chart = 'top200'
    GROUP BY region
    ORDER BY days_with_data ASC
    LIMIT 15
""").df()
print(result_with_days)

                  region  days_with_data
0             Luxembourg             297
1            South Korea             302
2                 Russia             504
3                Ukraine             504
4                  Egypt             787
5                Morocco             866
6           Saudi Arabia             912
7   United Arab Emirates             932
8                  India            1008
9              Nicaragua            1100
10              Bulgaria            1157
11          South Africa            1280
12               Romania            1358
13                Israel            1358
14               Vietnam            1358
